In [1]:
import pandas as pd
import numpy as np

In [7]:
%pip install surprise

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp312-cp312-win_amd64.whl size=1291318 sha256=91d5aadd4ced0142f0c8669388172b5bcc0808bcb817cbdb4a0076f2be796dfb
  Stored in directory: c:\users\joshua serrao\appdata\local\pip\cache\wheels\75\fa\bc\739bc2cb1fbaab6061854e6cfbb81a0ae52c92a502a7fa454b
Successfully built scikit-surprise
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
data_path = os.path.join(os.getcwd(), 'data')

In [28]:
prediction_data = pd.read_csv(os.path.join(data_path, 'prediction.csv'))
product_data = pd.read_json(os.path.join(data_path, 'product.json'))
review_data = pd.read_csv(os.path.join(data_path, 'review.csv'))
validation_data = pd.read_csv(os.path.join(data_path, 'validation.csv'))

print(f"Prediction data shape: {prediction_data.shape}")
print(f"Product data shape: {product_data.shape}")
print(f"Review data shape: {review_data.shape}")
print(f"Validation data shape: {validation_data.shape}")

Prediction data shape: (6633, 3)
Product data shape: (6309, 17)
Review data shape: (52512, 5)
Validation data shape: (6596, 3)


In [4]:
prediction_data

,ReviewerID,ProductID,Star
0,A2MK1L1Y74WTWH,B01GT5XDFS,0
1,A19I68RW4PBT29,B00OME9OQQ,0
2,A1UPHTDW5GM12T,B01GSRNLOK,0
3,A1LFIFPYMOJ8RV,B01CUJYMR0,0
4,A10Y597K071WTQ,B004SI455Q,0
...,...,...,...
6628,A23Y4UGTFDMZOP,B00J5327X6,0
6629,A2PFNDDKHOOMZU,B01G0GIXJ2,0
6630,A1K4S4MWXI9E9M,B01FKDKB96,0
6631,AOLHNMI8G8R6K,B00NUDPR66,0


In [5]:
review_data

,ReviewerID,ProductID,Text,Summary,Star
0,A1XJXYKOWCH9XT,B000FBFMHU,Liked the movie. Loved the book. It really giv...,Liked the movie. Loved the book!,5.0
1,A1K4S4MWXI9E9M,B000FC27TA,Purchased more out of curiosity than any real ...,"Not my favorite, but...",3.0
2,A3LF914GG87TWP,B000FC27TA,"I actually received this text as an ebook, sin...",An interesting read,4.0
3,A1CNQTCRQ35IMM,B000FCKPG2,REVIEWER'S OPINION:\r\nThis was labeled as rom...,This was labeled romance but there was less ro...,2.0
4,AU510CVD9XDG,B000GCFWXW,I have been saving the Argeneau novels for awh...,Science Fiction not Paranormal Romance,2.0
...,...,...,...,...,...
52507,A3JVZY05VLMYEM,B01FLJUZ0E,She can't do anything right according to her f...,What Can She Do,5.0
52508,A2U06P692IZOSF,B01FLJUZ0E,Better late than never!!\r\nKitty Konstantine ...,BART & KITTY CAT MAKE SPARKS FLY!!,5.0
52509,A3RPL8JIS2XMJ3,B01FLJUZ0E,This book was great. Bartholomew finally gets ...,LOVE THE SAINTS,5.0
52510,A1XMFCMIANCQRW,B01FPYJS1M,I read for a honest review for the author.\r\n...,"Loved Lee and Raina together, Ricky is evil an...",4.0


In [6]:
validation_data

,ReviewerID,ProductID,Star
0,A25X28UZCW2J6G,B00K9V6B94,4.0
1,A1FUH1O6FCTUYG,B00GZANS6M,5.0
2,AAUVEEG5YLZAX,B01864DDVO,5.0
3,A3VQLGTYTL5196,B001BXNQ2O,5.0
4,A10JAUCIGVRW9F,B0116MZUS2,5.0
...,...,...,...
6591,A3TC60MGLW1I76,B00EHSUFD8,4.0
6592,AGE0YGLF7L2ZL,B014LQ18CW,4.0
6593,AT2ZB20OCU7X2,B00ZRDPPU0,4.0
6594,A1ACUN6A2LYVMW,B01EKIELGG,1.0


## Using Surprise Library from Scikit-Learn

### Initial Algorithm

In [ ]:
import pandas as pd
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

train_df = review_data[['ReviewerID', 'ProductID', 'Star']].copy()

# Define the rating scale (1.0 to 5.0)
reader = Reader(rating_scale=(1.0, 5.0))

train_data = Dataset.load_from_df(train_df, reader)
trainset, testset = train_test_split(train_data, test_size=0.2)

model = SVD(n_factors=100, n_epochs=20, random_state=42)
model.fit(trainset)

# Evaluate on the local test set (optional)
predictions = model.test(testset)
rmse = accuracy.rmse(predictions)
print(f"Local RMSE: {rmse}")

val_df = validation_data
val_predictions = []
for _, row in val_df.iterrows():
    pred = model.predict(row['ReviewerID'], row['ProductID']).est
    val_predictions.append(pred)
val_df['Star'] = val_predictions
val_df.to_csv('val_prediction.csv', index=False)

import os
os.system('python evaluate.py val_prediction.csv')  

test_df = prediction_data
test_predictions = []
for _, row in test_df.iterrows():
    pred = model.predict(row['ReviewerID'], row['ProductID']).est
    test_predictions.append(pred)
test_df['Star'] = test_predictions
test_df.to_csv('prediction_test.csv', index=False)  # Overwrite with predictions


RMSE: 0.7519
Local RMSE: 0.7519116751866977


### Hyperparameter Tuning

In [36]:
from surprise.model_selection import GridSearchCV

param_grid = {
    'n_factors': [50, 100, 150],
    'n_epochs': [50, 100],
    'lr_all': [0.002, 0.01, 0.001, 0.005],
    'reg_all': [0.02, 0.1, 0.5]
}
gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3)
gs.fit(train_data)
print(gs.best_params['rmse'])
# model = SVD(**gs.best_params['rmse'])
# model.fit(trainset)

{'n_factors': 50, 'n_epochs': 50, 'lr_all': 0.005, 'reg_all': 0.1}


### Model Testing after Hyperparameter Tuning

In [37]:
train_df = review_data

reader = Reader(rating_scale=(1.0, 5.0))
train_data = Dataset.load_from_df(train_df[['ReviewerID', 'ProductID', 'Star']], reader)

# Split for local validation
trainset = train_data.build_full_trainset()

model = SVD(
    n_factors=50,    
    n_epochs=50,     
    lr_all=0.005,  
    reg_all=0.1,     
    random_state=42  
)
model.fit(trainset)  

# Evaluate on local test set (optional)
# predictions = model.test(testset)
# rmse = accuracy.rmse(predictions)
# print(f"Local RMSE: {rmse}")

val_df = validation_data
val_predictions = []
for _, row in val_df.iterrows():
    pred = model.predict(row['ReviewerID'], row['ProductID']).est
    val_predictions.append(pred)
val_df['Star'] = val_predictions
val_df.to_csv('validation_prediction_tuned.csv', index=False)

# import os
# os.system('python evaluate.py val_prediction2.csv')

# test_df = prediction_data
# test_predictions = []
# for _, row in test_df.iterrows():
#     pred = model.predict(row['ReviewerID'], row['ProductID']).est
#     test_predictions.append(pred)
# test_df['Star'] = test_predictions
# test_df.to_csv('prediction_test2.csv', index=False)

# print("Predictions saved to prediction_test2.csv")

In [15]:
validation_data

In [38]:
import pandas as pd
from surprise import SVD, Dataset, Reader, SVDpp
from surprise.model_selection import train_test_split
from surprise import accuracy

train_df = review_data[['ReviewerID', 'ProductID', 'Star']].copy()

# Define the rating scale (1.0 to 5.0)
reader = Reader(rating_scale=(1.0, 5.0))

train_data = Dataset.load_from_df(train_df, reader)
# val_data = Dataset.load_from_df(validation_data[['ReviewerID', 'ProductID']], reader)
# val_test_set = val_data.build_testset()

# # trainset, testset = train_test_split(train_data, test_size=0.2)
trainset = train_data.build_full_trainset()

model = SVDpp(n_factors=100, n_epochs=50, random_state=42)
model.fit(trainset)
# val_test_set = val_data.build_testset()

# # Evaluate on the validation set
# predictions = model.predict(val_data)

# rmse = accuracy.rmse(predictions)
# print(f"Local RMSE: {rmse}")

val_df = validation_data
val_predictions = []
for _, row in val_df.iterrows():
    pred = model.predict(row['ReviewerID'], row['ProductID']).est
    val_predictions.append(pred)
val_df['Star'] = val_predictions
val_df.to_csv('validation_prediction_3.csv', index=False)

# import os
# os.system('python evaluate.py val_prediction.csv')  

# test_df = prediction_data
# test_predictions = []
# for _, row in test_df.iterrows():
#     pred = model.predict(row['ReviewerID'], row['ProductID']).est
#     test_predictions.append(pred)
# test_df['Star'] = test_predictions
# test_df.to_csv('prediction_test.csv', index=False)  # Overwrite with predictions


In [22]:
train_data

In [ ]:
val_data = Dataset.load_from_df(validation_data, reader)


AttributeError: 'DatasetAutoFolds' object has no attribute 'itertuples'

In [18]:
train_df

,ReviewerID,ProductID,Star
0,A1XJXYKOWCH9XT,B000FBFMHU,5.0
1,A1K4S4MWXI9E9M,B000FC27TA,3.0
2,A3LF914GG87TWP,B000FC27TA,4.0
3,A1CNQTCRQ35IMM,B000FCKPG2,2.0
4,AU510CVD9XDG,B000GCFWXW,2.0
...,...,...,...
52507,A3JVZY05VLMYEM,B01FLJUZ0E,5.0
52508,A2U06P692IZOSF,B01FLJUZ0E,5.0
52509,A3RPL8JIS2XMJ3,B01FLJUZ0E,5.0
52510,A1XMFCMIANCQRW,B01FPYJS1M,4.0


In [19]:
validation_data


,ReviewerID,ProductID,Star
0,A25X28UZCW2J6G,B00K9V6B94,4.0
1,A1FUH1O6FCTUYG,B00GZANS6M,5.0
2,AAUVEEG5YLZAX,B01864DDVO,5.0
3,A3VQLGTYTL5196,B001BXNQ2O,5.0
4,A10JAUCIGVRW9F,B0116MZUS2,5.0
...,...,...,...
6591,A3TC60MGLW1I76,B00EHSUFD8,4.0
6592,AGE0YGLF7L2ZL,B014LQ18CW,4.0
6593,AT2ZB20OCU7X2,B00ZRDPPU0,4.0
6594,A1ACUN6A2LYVMW,B01EKIELGG,1.0


In [41]:
from surprise.model_selection import GridSearchCV, RandomizedSearchCV

param_grid = {
    'n_factors': [50, 100, 150],
    'n_epochs': [50, 100],
    'lr_all': [0.002, 0.01, 0.001, 0.005],
    'reg_all': [0.02, 0.1, 0.5]
}
gs = RandomizedSearchCV(SVDpp, param_grid, measures=['rmse'], cv=5, n_iter=15)
gs.fit(train_data)
print(gs.best_params['rmse'])

{'n_factors': 50, 'n_epochs': 50, 'lr_all': 0.005, 'reg_all': 0.1}


In [40]:
train_df = review_data

reader = Reader(rating_scale=(1.0, 5.0))
train_data = Dataset.load_from_df(train_df[['ReviewerID', 'ProductID', 'Star']], reader)

# Split for local validation
trainset = train_data.build_full_trainset()

model = SVDpp(
    n_factors=50,    
    n_epochs=50,     
    lr_all=0.005,  
    reg_all=0.1,     
    random_state=42  
)
model.fit(trainset)  

# Evaluate on local test set (optional)
# predictions = model.test(testset)
# rmse = accuracy.rmse(predictions)
# print(f"Local RMSE: {rmse}")

val_df = validation_data
val_predictions = []
for _, row in val_df.iterrows():
    pred = model.predict(row['ReviewerID'], row['ProductID']).est
    val_predictions.append(pred)
val_df['Star'] = val_predictions
val_df.to_csv('validation_prediction_tuned_randomized_svdpp.csv', index=False)